In [0]:
%sql
create schema if not exists workspace.ecommerce;
create volume if not exists workspace.ecommerce.raw_files;

In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/ecommerce/raw_files"
SCHEMA = "workspace.ecommerce"

files = {
    "bronze_orders" : "olist_orders_dataset.csv",
    "bronze_order_items": "olist_order_items_dataset.csv",
    "bronze_order_payments": "olist_order_payments_dataset.csv",
    "bronze_order_reviews": "olist_order_reviews_dataset.csv",
    "bronze_customers": "olist_customers_dataset.csv",
    "bronze_products": "olist_products_dataset.csv",
    "bronze_sellers": "olist_sellers_dataset.csv",
    "bronze_category_translation": "product_category_name_translation.csv",
}

for table, fname in files.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape",'"')
          .csv(f"{BASE}/{fname}")
          .withColumn("_source_file", F.lit(fname))
          .withColumn("_ingested_at", F.current_timestamp())
          )
    df.write.mode("overwrite").saveAsTable(f"{SCHEMA}.{table}")
    print(table, df.count())



In [0]:
spark.sql("SHOW TABLES IN workspace.ecommerce").show()
spark.table("workspace.ecommerce.bronze_orders").show(5, truncate=False)